# The Liar's Chatroom: methodology notebook

**Final submission:** `submission_v4_th0.8.csv` (v4), public 0.99444, private 0.99192.

**Model:** Qwen3-14B, 4-bit AWQ ([Qwen/Qwen3-14B-AWQ](https://huggingface.co/Qwen/Qwen3-14B-AWQ)), 14.8B parameters, Apache-2.0. Inference is local with vLLM on Kaggle GPU T4 x2. No hosted APIs, no fine-tuning, no external data.

**Compliance:** test rows were never printed, read or hand-labelled, and no rule was designed from test.csv. Every decision policy was validated on train only.

Pipeline:
1. Exact rule modules (100% precision on train)
2. Qwen3-14B thinking-mode reasoning
3. Hard-case refinement with self-consistency
4. Second opinions and rule checks by voting

Place the competition data in `Data/` (train.csv, test.csv, sample_submission.csv).

In [ ]:
# Setup (GPU needed only for the LLM cells)
# !pip install -q vllm pandas
import sys, collections, pandas as pd
sys.path.insert(0, '.')
from liars.rules import solve
from liars.prompt import build_prompt, extract_label
train = pd.read_csv('Data/train.csv')
print(train.shape)

## 1. Exact rule modules (`liars/rules.py`)
There are 27 parsers, one per evidence family: time windows, vehicle round trips, ages, rotas, tank levels, bookings, ownership chains and others. A module flags a message only when it can prove the message contradicts the evidence. `solve` returns a label only when every flag points to one sender. When the evidence contains unfamiliar qualifiers (except, unless, if, or, ...), the rules abstain and the LLM decides, which protects against scenario families not seen in train.

In [ ]:
preds = [solve(r)[0] for _, r in train.iterrows()]
covered = [(p, l) for p, l in zip(preds, train['label']) if p]
print(f"rule coverage {len(covered)}/{len(train)} = {len(covered)/len(train):.1%}, "
      f"precision {sum(p == l for p, l in covered)}/{len(covered)}")

## 2. LLM reasoning (Qwen3-14B, thinking mode)
The prompt states the competition's contradiction rules, drops questions and small talk, labels each message with the sender's name, and asks for `ANSWER: <message number>`, which is mapped to its sender. Sampling: temperature 0.6, top_p 0.95, top_k 20. Truncated thinking is budget-forced into an answer.

In [ ]:
print(build_prompt(train.iloc[0])[:2500])

In [ ]:
RUN_LLM = False  # set True on a GPU machine (Kaggle T4 x2)
FORCE = ("\n\nConsidering the limited time, I have to give the answer based on the thinking directly now.\n"
         "</think>\n\nANSWER:")

def llm_votes(llm, df, k, max_tokens, seed=0):
    from vllm import SamplingParams
    tok = llm.get_tokenizer()
    chats = [tok.apply_chat_template([{"role": "user", "content": build_prompt(r)}], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=True) for _, r in df.iterrows()]
    outs = llm.generate(chats, SamplingParams(n=k, temperature=0.6, top_p=0.95, top_k=20,
                                              max_tokens=max_tokens, seed=seed))
    votes, force = [], []
    for j, ((_, r), o) in enumerate(zip(df.iterrows(), outs)):
        votes.append([])
        for c in o.outputs:
            lab = extract_label(c.text, r) if "</think>" in c.text else None
            if lab: votes[j].append(lab)
            else: force.append((j, r, chats[j] + c.text + FORCE))
    if force:
        fo = llm.generate([p for *_, p in force], SamplingParams(temperature=0.0, max_tokens=12))
        for (j, r, _), o in zip(force, fo):
            lab = extract_label("ANSWER:" + o.outputs[0].text, r)
            if lab: votes[j].append(lab)
    return votes

if RUN_LLM:
    from vllm import LLM
    llm = LLM(model="Qwen/Qwen3-14B-AWQ", tensor_parallel_size=2, dtype="half", max_model_len=12288,
              gpu_memory_utilization=0.92, enable_prefix_caching=True, seed=0)
    sample = train[[p is None for p in preds]].sample(20, random_state=0)
    v = llm_votes(llm, sample, k=1, max_tokens=6000)
    print("accuracy", sum(x[:1] == [l] for x, l in zip(v, sample['label'])) / len(sample))

## 3. Hard-case refinement (`kaggle/refine/main.py`)
Cases whose first reasoning ran to 2,000 tokens or more, or produced no answer, get 2 more samples with a 10k-token budget. The final label is a 2-of-3 majority including the first answer. On the matching train cases this went from 38/40 to 40/40.

## 4. Second opinions and rule checks (`kaggle/verify/main.py`)
Every single-sample LLM case and every rule-covered case gets one more sample. Disagreements get 3 more samples, and the 49 most contested cases get 6 more. Final decision policy:

In [ ]:
def decide_llm_case(current, votes):
    """Majority over every vote collected; a tie keeps the current label."""
    c = collections.Counter(votes)
    best = max(c.values())
    tops = [k for k in c if c[k] == best]
    return current if current in tops else (tops[0] if len(tops) == 1 else current)

def decide_rule_case(rule_label, llm_votes, threshold=0.8):
    """Override an exact rule only when >= 80% of the LLM votes agree on another sender.
    Train: 7 of 10 votes against a correct rule still occurred, so 0.7 would break a correct rule."""
    against = [x for x in llm_votes if x != rule_label]
    if against:
        lab, n = collections.Counter(against).most_common(1)[0]
        if n / len(llm_votes) >= threshold:
            return lab
    return rule_label

print(decide_llm_case('A', ['A', 'B', 'B']), decide_rule_case('C', list('BCBBBBCBCB')))

## Results
| Version | Public | Private |
|---|---|---|
| v1 rules + Qwen3-14B | 0.98777 | n/a |
| v2b hard-case refinement | 0.99222 | 0.99038 |
| v3a second opinions, majority vote | 0.99222 | 0.99192 |
| **v4 + extra votes and rule overrides at 80%** | **0.99444** | **0.99192** |

Full reproduction: `scripts/build_kernel.py --mode test` → `scripts/build_refine.py` → `scripts/build_verify.py` → `scratch/combine_v4.py 0.8` (each Kaggle kernel reads the previous kernel's outputs).